In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# --- Constants ---
mu = 3.67537  # reduced mass in amu
prefactor = 3.7313e7 / np.sqrt(mu)

# --- User input: Energies (MeV) and Eσ(E) (MeV·barn) ---
E = np.array(list(map(float, input("Enter E values in MeV: ").split())))
E_sigma_E = np.array(list(map(float, input("Enter Eσ(E) values in MeV·barn: ").split())))

# Convert Eσ(E) from barns → millibarns (multiply by 1000)
E_sigma_E_mb = E_sigma_E * 1000.0

# Create interpolation function (linear, no oscillations)
interp_func = interp1d(E, E_sigma_E_mb, kind="linear", fill_value="extrapolate")

# --- Gauss–Legendre Reaction Rate function ---
def reaction_rate_gauss(T9, E_min, E_max, interp_func, n_points=64):
    # Gauss–Legendre nodes & weights
    x, w = np.polynomial.legendre.leggauss(n_points)
    # Transform to [E_min, E_max]
    E_nodes = 0.5*(E_max-E_min)*x + 0.5*(E_max+E_min)
    w_nodes = 0.5*(E_max-E_min)*w

    # Interpolated values (clamp negatives to zero)
    E_sigma_vals = interp_func(E_nodes)
    E_sigma_vals = np.where(E_sigma_vals < 0, 0.0, E_sigma_vals)

    # Integrand for rate
    integrand = E_sigma_vals * np.exp(-11.605 * E_nodes / T9)

    # Integration
    integral = np.sum(w_nodes * integrand)

    return prefactor * T9**(-1.5) * integral

# --- Temperatures (GK) ---
T9_vals = [0.001,0.225, 0.45, 0.674, 0.899, 1.123, 1.348, 1.572, 1.797, 2.021, 2.246,
    2.47, 2.695, 2.919, 3.144, 3.368, 3.593, 3.817, 4.041, 4.266, 4.49,
    4.715, 4.939, 5.164, 5.388, 5.613, 5.837, 6.062, 6.286, 6.511, 6.735,
    6.96, 7.184, 7.408, 7.633, 7.857, 8.082, 8.306, 8.531, 8.755, 8.98,
    9.204, 9.429, 9.653, 9.878, 10.102, 10.327, 10.551, 10.776, 11.0
]

# --- Compute & print ---
print(f"{'T9 (GK)':>8} {'Rate (cm³ mol⁻¹ s⁻¹)':>25} {'log10(rate)':>15}")
print("-"*55)

for T9 in T9_vals:
    rate = reaction_rate_gauss(T9, np.min(E), np.max(E), interp_func, n_points=64)
    rate = max(rate, 0.0)   # Final safeguard against negatives
    log_rate = np.log10(rate) if rate > 0 else -np.inf
    print(f"{T9:8.3f} {rate:25.6e} {log_rate:15.4f}")


Enter E values in MeV: 0 0.096 0.192 0.288 0.384 0.48 0.575 0.671 0.767 0.863 0.959 1.055 1.151 1.247 1.343 1.439 1.534 1.63 1.726 1.822 1.918 2.014 2.11 2.206 2.302 2.398 2.493 2.589 2.685 2.781 2.877 2.973 3.069 3.165 3.261 3.357 3.453 3.548 3.644 3.74 3.836 3.932 4.028 4.124 4.22 4.316 4.412 4.507 4.603 4.699 4.795 4.891 4.987 5.083 5.179 5.275 5.371 5.466 5.562 5.658 5.754 5.85 5.946 6.042 6.138 6.234 6.33 6.426 6.521 6.617 6.713 6.809 6.905 7.001 7.097 7.193 7.289 7.385 7.48 7.576 7.672 7.768 7.864 7.96 8.056 8.152 8.248 8.344 8.44 8.535 8.631 8.727 8.823 8.919 9.015 9.111
Enter Eσ(E) values in MeV·barn: 0.00E+00 4.34E-99 2.54E-66 8.73E-52 4.18E-43 3.58E-37 8.74E-33 2.28E-29 1.30E-26 2.53E-24 2.20E-22 1.03E-20 2.96E-19 5.74E-18 8.08E-17 8.71E-16 7.49E-15 5.32E-14 3.20E-13 1.67E-12 7.64E-12 3.14E-11 1.17E-10 3.98E-10 1.25E-09 3.67E-09 1.01E-08 2.61E-08 6.40E-08 1.49E-07 3.33E-07 7.13E-07 1.47E-06 2.91E-06 5.59E-06 1.04E-05 1.88E-05 3.30E-05 5.65E-05 9.44E-05 1.54E-04 2.46E-04 3.86E